# DriveSafe AI
Driver Risk Score Analytics System

In [ ]:
print('DriveSafe AI notebook initialized')

In [1]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "data").exists() else NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
VISUALS_DIR = PROJECT_ROOT / "visuals"

for folder in [DATA_DIR, MODELS_DIR, REPORTS_DIR, VISUALS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


def generate_driver_dataset(n_rows: int = 5000, random_state: int = 42) -> pd.DataFrame:
    local_rng = np.random.default_rng(random_state)

    driver_id = np.arange(100001, 100001 + n_rows)
    age = np.clip(local_rng.normal(39, 11, n_rows), 18, 70).round().astype(int)
    experience_years = np.clip(
        age - 18 - local_rng.normal(1.8, 4.0, n_rows), 0, None
    ).round().astype(int)
    avg_speed = np.clip(local_rng.normal(58, 12, n_rows) + local_rng.normal(0, 4, n_rows), 25, 110)
    harsh_braking = local_rng.poisson(
        lam=np.clip((avg_speed - 40) / 18 + local_rng.normal(0.8, 0.4, n_rows), 0.2, 6)
    )
    sudden_acceleration = local_rng.poisson(
        lam=np.clip((avg_speed - 36) / 20 + local_rng.normal(0.7, 0.4, n_rows), 0.2, 5)
    )
    weekly_driving_hours = np.clip(local_rng.normal(38, 12, n_rows) + 0.22 * experience_years, 5, 85)
    mobile_usage_minutes = np.clip(
        local_rng.gamma(shape=2.2, scale=8.5, size=n_rows) + np.maximum(0, 12 - experience_years * 0.15),
        0,
        130,
    )
    night_driving_hours = np.clip(
        local_rng.normal(3.0, 1.7, n_rows) + np.maximum(0, 28 - experience_years) * 0.03, 0, 12
    )
    fatigue_score = np.clip(
        0.24 * night_driving_hours + 0.014 * weekly_driving_hours + 0.028 * mobile_usage_minutes
        + local_rng.normal(1.8, 1.1, n_rows),
        0,
        10,
    )
    traffic_violations = local_rng.poisson(
        lam=np.clip(0.08 * harsh_braking + 0.1 * sudden_acceleration + 0.05 * night_driving_hours, 0.05, 5)
    )
    seatbelt_compliance = np.clip(
        local_rng.normal(0.93, 0.07, n_rows)
        - 0.0015 * mobile_usage_minutes
        - 0.001 * night_driving_hours
        + 0.001 * experience_years,
        0.35,
        1.0,
    )
    weather_risk = local_rng.choice([1, 2, 3, 4, 5], size=n_rows, p=[0.17, 0.26, 0.28, 0.19, 0.10])
    accident_history = local_rng.poisson(
        lam=np.clip(0.06 * traffic_violations + 0.03 * night_driving_hours + 0.02 * weather_risk, 0.0, 3.0)
    )

    latent_risk = (
        0.30 * (avg_speed - 50) / 10
        + 0.42 * harsh_braking
        + 0.36 * sudden_acceleration
        + 0.06 * mobile_usage_minutes
        + 0.60 * night_driving_hours
        + 0.72 * fatigue_score
        + 0.58 * traffic_violations
        + 2.2 * (1 - seatbelt_compliance)
        + 0.48 * weather_risk
        + 0.05 * weekly_driving_hours
        + 1.05 * accident_history
        - 0.06 * experience_years
        - 0.03 * np.maximum(0, age - 18)
        + local_rng.normal(0, 2.0, n_rows)
    )

    low_cutoff, high_cutoff = np.quantile(latent_risk, [0.56, 0.82])
    driver_risk = np.where(
        latent_risk < low_cutoff,
        "Safe",
        np.where(latent_risk < high_cutoff, "Moderate Risk", "High Risk"),
    )

    df = pd.DataFrame(
        {
            "driver_id": driver_id,
            "age": age,
            "experience_years": experience_years,
            "avg_speed": avg_speed.round(2),
            "harsh_braking": harsh_braking,
            "sudden_acceleration": sudden_acceleration,
            "mobile_usage_minutes": mobile_usage_minutes.round(2),
            "night_driving_hours": night_driving_hours.round(2),
            "fatigue_score": fatigue_score.round(2),
            "traffic_violations": traffic_violations,
            "seatbelt_compliance": seatbelt_compliance.round(3),
            "weather_risk": weather_risk,
            "weekly_driving_hours": weekly_driving_hours.round(2),
            "accident_history": accident_history,
            "driver_risk": driver_risk,
        }
    )

    missing_specs = {
        "mobile_usage_minutes": 0.035,
        "night_driving_hours": 0.03,
        "seatbelt_compliance": 0.025,
        "traffic_violations": 0.02,
        "weather_risk": 0.02,
    }
    for column, fraction in missing_specs.items():
        missing_index = local_rng.choice(df.index, size=int(n_rows * fraction), replace=False)
        df.loc[missing_index, column] = np.nan

    return df


In [2]:
driver_df = generate_driver_dataset()
dataset_path = DATA_DIR / "driver_risk_dataset.csv"
driver_df.to_csv(dataset_path, index=False)

print(f"Saved synthetic dataset to: {dataset_path}")
print(f"Rows: {driver_df.shape[0]}, Columns: {driver_df.shape[1]}")
print(driver_df["driver_risk"].value_counts(normalize=True).sort_index().round(3))
driver_df.head()


Saved synthetic dataset to: d:\Driver Risk Score Analysis\data\driver_risk_dataset.csv
Rows: 5000, Columns: 15
driver_risk
High Risk        0.18
Moderate Risk    0.26
Safe             0.56
Name: proportion, dtype: float64


,driver_id,age,experience_years,avg_speed,harsh_braking,sudden_acceleration,mobile_usage_minutes,night_driving_hours,fatigue_score,traffic_violations,seatbelt_compliance,weather_risk,weekly_driving_hours,accident_history,driver_risk
0,100001,42,23,63.83,1,1,64.84,3.37,4.50,0.0,NaN,2.0,53.59,0,Safe
1,100002,28,9,72.63,5,1,23.14,0.00,3.09,1.0,0.889,4.0,62.82,0,Moderate Risk
2,100003,47,30,43.33,1,1,18.26,2.59,4.72,0.0,0.909,3.0,61.99,1,Safe
3,100004,49,27,56.34,2,2,39.08,1.32,3.71,0.0,0.749,1.0,45.45,0,Safe
4,100005,18,0,69.60,2,2,22.04,3.49,3.57,0.0,0.914,3.0,43.77,1,High Risk


In [3]:
driver_df = pd.read_csv(DATA_DIR / "driver_risk_dataset.csv")
print(f"Loaded dataset shape: {driver_df.shape}")
display(driver_df.head())
display(driver_df.sample(5, random_state=RANDOM_STATE))
print(driver_df.dtypes)
print(driver_df["driver_risk"].value_counts())


Loaded dataset shape: (5000, 15)


,driver_id,age,experience_years,avg_speed,harsh_braking,sudden_acceleration,mobile_usage_minutes,night_driving_hours,fatigue_score,traffic_violations,seatbelt_compliance,weather_risk,weekly_driving_hours,accident_history,driver_risk
0,100001,42,23,63.83,1,1,64.84,3.37,4.50,0.0,NaN,2.0,53.59,0,Safe
1,100002,28,9,72.63,5,1,23.14,0.00,3.09,1.0,0.889,4.0,62.82,0,Moderate Risk
2,100003,47,30,43.33,1,1,18.26,2.59,4.72,0.0,0.909,3.0,61.99,1,Safe
3,100004,49,27,56.34,2,2,39.08,1.32,3.71,0.0,0.749,1.0,45.45,0,Safe
4,100005,18,0,69.60,2,2,22.04,3.49,3.57,0.0,0.914,3.0,43.77,1,High Risk


,driver_id,age,experience_years,avg_speed,harsh_braking,sudden_acceleration,mobile_usage_minutes,night_driving_hours,fatigue_score,traffic_violations,seatbelt_compliance,weather_risk,weekly_driving_hours,accident_history,driver_risk
1501,101502,32,6,69.44,5,0,44.56,NaN,4.46,1.0,0.836,4.0,41.63,0,High Risk
2586,102587,40,17,54.99,2,5,20.61,3.23,4.75,0.0,0.803,3.0,28.25,0,Safe
2653,102654,30,10,65.29,3,2,50.59,4.58,3.76,0.0,0.964,NaN,47.44,0,High Risk
1055,101056,55,33,80.39,2,1,20.77,0.17,2.25,0.0,0.874,2.0,49.83,0,Safe
705,100706,28,7,54.25,0,1,22.75,6.98,3.81,0.0,0.794,1.0,54.99,0,Safe


driver_id                 int64
age                       int64
experience_years          int64
avg_speed               float64
harsh_braking             int64
sudden_acceleration       int64
mobile_usage_minutes    float64
night_driving_hours     float64
fatigue_score           float64
traffic_violations      float64
seatbelt_compliance     float64
weather_risk            float64
weekly_driving_hours    float64
accident_history          int64
driver_risk                 str
dtype: object
driver_risk
Safe             2800
Moderate Risk    1300
High Risk         900
Name: count, dtype: int64


In [ ]:
print(f"Dataset shape: {driver_df.shape}")
print("Duplicate rows:", driver_df.duplicated().sum())
print("Target distribution:\n", driver_df["driver_risk"].value_counts())

display(driver_df.describe(include="all").T)
